In [0]:
 #%pip install xgboost==2.0.3 plotly workalendar holidays
 %pip install xgboost==2.0.3 plotly workalendar holidays hyperopt mlflow scikit-learn xgboost


In [0]:
#Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import xgboost as xgb
import mlflow
import mlflow.xgboost
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from datetime import timedelta
from pyspark.sql import SparkSession
from workalendar.america.brazil import BrazilSaoPauloCity

In [0]:
# 1. Ingestão Nativa do Spark Connector - 100x mais rápido internamente
print("Loading Gold layer from Lakehouse...")
try:
    df_spark_table = spark.table("foodshop_consumption")
    df = df_spark_table.toPandas()
except Exception as e:
    print("FATAL ERROR: Delta Table 'foodshop_consumption' not found! Make sure to run the ETL Script first.")
    raise e

# Mapeia as raízes do tempo
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True, drop=False)
df.sort_index(inplace=True)

Loading Gold layer from Lakehouse...


In [0]:

def build_features(df_input):
    """ Constrói de forma algorítmica D-1, Rolling Means e Tratativas de Calendário """
    dff = df_input.copy()
    
    # Quebra atômica de Séries Temporais (Sazonalidades)
    dff['day_of_week'] = dff['date'].dt.dayofweek
    dff['day_of_year'] = dff['date'].dt.dayofyear
    dff['month']       = dff['date'].dt.month
    dff['year']        = dff['date'].dt.year
    
    # Imputação Estatística Sazonal de Lacunas 
    # Protege a série preenchendo furos com a média do respectivo mês/ano!
    # Verifica se a coluna existe; preenche NaNs com a média do mesmo ano/mês
    # e depois usa propagação de valores (para frente e para trás) para eliminar valores restantes
    if 'consumption_mwh_day' in dff.columns:
        dff['consumption_mwh_day'] = dff['consumption_mwh_day'].fillna(dff.groupby(['year', 'month'])['consumption_mwh_day'].transform('mean'))
        dff['consumption_mwh_day'] = dff['consumption_mwh_day'].ffill().bfill() # bfill pega valores sem ser Nan anteriores e ffill faz o oposto
        
    if 'temp_mean_c' in dff.columns:
        dff['temp_mean_c'] = dff['temp_mean_c'].fillna(dff.groupby(['year', 'month'])['temp_mean_c'].transform('mean'))
        dff['temp_mean_c'] = dff['temp_mean_c'].ffill().bfill() # Backup

    # caso tenha valores ausentes preenche com 0 - nao feriado
    dff['is_holiday'] = dff['is_holiday'].fillna(0).astype(int)
    
    # Criação do Alvo (Target y) estabilizado usando transformação Logarítmica Neperiana !
    dff['y_log'] = np.log(dff['consumption_mwh_day'])
    
    # Auto-Regressão Lags
    dff['lag_1'] = dff['y_log'].shift(1)  # Ontem
    dff['lag_7'] = dff['y_log'].shift(7)  # Ontem na semana passada
    
    # Suavização Rolling Mean em Cima do Log
    dff['rolling_mean_7'] = dff['y_log'].rolling(window=7).mean()
    
    return dff

print("Cooking Matrix Feature Engineering...")
# Passa o pipeline
df_features = build_features(df)

# Mapeamento do Core Computacional Base 
FEATURES = [
    'temp_mean_c', 'is_holiday', 'day_of_week', 
    'day_of_year', 'month', 'year', 
    'lag_1', 'lag_7', 'rolling_mean_7'
]
TARGET = 'y_log'

# O Lags gera valores NaN na primeira semana de 2019 que precisamos decapotar para treinar com rigor.
# Adicionamos também uma varredura absoluta contra Infinitos (do np.log) ou buracos brutos da extração ONS.
# Substitui valores infinitos (positivos e negativos) por NaN para evitar problemas em análises/modelos
df_features = df_features.replace([np.inf, -np.inf], np.nan)
# Remove linhas com valores ausentes nas colunas usadas no modelo (features + target)
# e cria uma cópia limpa; depois reseta o índice do DataFrame
df_ml = df_features.dropna(subset=FEATURES + [TARGET]).copy()
df_ml = df_ml.reset_index(drop=True)





# Subdivisão Cronológica Perfeita sem Data Leakage
train_size = len(df_ml) - 90
#Ordem cronologica importa para series temporais
train_df = df_ml.iloc[:train_size]
test_df  = df_ml.iloc[train_size:]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]


Cooking Matrix Feature Engineering...


In [0]:

def train_and_optimize():
    print(f"\n[HYPEROPT] Distributing Hyperparameter testing on the cluster (Train: {len(X_train)} d | Validation: {len(X_test)} d)...")
    
    # Limites setados para parametros
    search_space = {
        # Usa funções do Hyperopt (hp) para definir como os hiperparâmetros serão sorteados/testados durante a otimização

        'max_depth': hp.quniform('max_depth', 3, 9, 1),
        'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
        'subsample': hp.uniform('subsample', 0.6, 1.0),
        'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
        'n_estimators': hp.quniform('n_estimators', 100, 1000, 100)
    }
    
    def objective(params):
        # Converte parâmetros discretos de float para int (necessário para o modelo)
        params['max_depth'] = int(params['max_depth'])
        params['n_estimators'] = int(params['n_estimators'])
        
        # Abastece localmente na árvore MLflow do Painel
        # Inicia um experimento aninhado no MLflow para rastrear essa tentativa
        with mlflow.start_run(nested=True): #serve para organizar múltiplas tentativas dentro de um mesmo experimento de forma estruturada
            # Cria e treina o modelo XGBoost com os parâmetros testados
            model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1, eval_metric='rmse')
            model.fit(X_train, y_train)
            
            # Faz previsões no conjunto de teste
            preds_log = model.predict(X_test)
            
            # --- CONVERSÃO INVERSA ANTES DAS MÉTRICAS ---
            # Transforma a resposta e o valore real (exp) de volta à escala original (R$/MWh)
            preds_real = np.exp(preds_log)
            y_test_real = np.exp(y_test)
            
            # Calcula métricas de erro na escala real (negócio)
            rmse = np.sqrt(mean_squared_error(y_test_real, preds_real))
            mae = mean_absolute_error(y_test_real, preds_real)
            
            # Registra parâmetros e métricas no MLflow
            mlflow.log_params(params)
            mlflow.log_metric("rmse_mwh", rmse)
            mlflow.log_metric("mae_mwh", mae)
            # Retorna o erro para o Hyperopt (que tenta minimizar esse valor)
            return {'loss': rmse, 'status': STATUS_OK}

    # Utiliza-se executor em Driver Normal (Pois Clusters Serverless bloqueiam o JVM do SparkTrials)
    # Inicializa objeto que armazena os resultados das tentativas
    trials = Trials() #controle interno da otimização do Hyperopt - hp
    # Executa a busca pelos melhores hiperparâmetros
    best_params = fmin(
        fn=objective, # função que avalia cada tentativa
        space=search_space,  # espaço de busca
        algo=tpe.suggest, # algoritmo de otimização (Bayesiano)
        max_evals=15,  # NUmeros de testes
        trials=trials
    )
    # Ajusta tipos dos parâmetros finais
    best_params['max_depth'] = int(best_params['max_depth'])
    best_params['n_estimators'] = int(best_params['n_estimators'])
    # Mostra os melhores parâmetros encontrados
    print(f"[WINNING HYPEROPT SCORE] -> Champion Parameters: {best_params}")
    return best_params
    #End function




[HYPEROPT] Distributing Hyperparameter testing on the cluster (Train: 2546 d | Validation: 90 d)...
100%|██████████| 15/15 [00:11<00:00,  1.34trial/s, best loss: 12.470904191455407]
[WINNING HYPEROPT SCORE] -> Champion Parameters: {'colsample_bytree': np.float64(0.9625209633783741), 'learning_rate': np.float64(0.2423665521995597), 'max_depth': 8, 'n_estimators': 100, 'subsample': np.float64(0.7442600491100743)}

(REFIT) Re-training final unstoppable model on 100% full temporal timeline...


2026/03/28 18:23:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-41cdb982-37e6.cloud.databricks.com/ml/experiments/470293763348821/models/m-07e9a43b4e7d4ccd83defe38c7ff248b?o=7474658862847997
2026/03/28 18:24:00 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-c7d0d651-3092-429c-91dc-d8/tmpbeo8ofal/model, flavor: xgboost). Fall back to return ['xgboost==2.0.3']. Set logging level to DEBUG to see the full traceback. 
2026/03/28 18:24:00 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signature

[LOG] Physical tree successfully saved on Databricks Container.


In [ ]:
# Define onde os experimentos serão salvos no MLflow
mlflow.set_experiment("/Shared/XGBoost_Energy_Prescriptive")
# Inicia execução principal e roda a otimização
with mlflow.start_run(run_name="Best_Architected_XGBoost"):
    best_hypervars = train_and_optimize()



# Treina o modelo final usando 100% dos dados disponíveis

print("\n(REFIT) Re-training final unstoppable model on 100% full temporal timeline...")
X_full = df_ml[FEATURES]
y_full = df_ml[TARGET]

final_model = xgb.XGBRegressor(**best_hypervars, random_state=42, n_jobs=-1)
final_model.fit(X_full, y_full)

# Guarda fisicamente essa instância no MLFlow (Permite o versionamento nativo com 1 clique)
mlflow.xgboost.log_model(final_model, artifact_path="model_xgboost_prescriptive")
print("[LOG] Physical tree successfully saved on Databricks Container.")





In [0]:
#---------PREVISAO DOS PROXIMOS MESES (AINDA SEM DADOS REAIS)-------------
# Para calcular dinâmicamente precisamos empurrar o tempo "mês a mês - dia a dia" e o modelo responder em loop
cal = BrazilSaoPauloCity()
def run_recursive_future(model, df_hist, days_forward=90):
    print(f"\nInitiating Auto-Regressive recursive chained prediction matrix (Next {days_forward} days)...")
    
    last_known_date = df_hist['date'].iloc[-1] #Ultimo dia conhecido
    
    # Histórico de Log-consumo cru do array original (necessário para buscar Lag-7 dinamicamente)
    hist_logs = list(df_hist['y_log'].values)
    
    # media temp mensal
    temp_annual_means = df_hist.groupby('month')['temp_mean_c'].mean().to_dict()
    
    predictions = []
    future_dates = []
    
    date_pointer = last_known_date
    for day in range(days_forward):
        # Avança 1 dia
        date_pointer += timedelta(days=1) #timedelta é um objeto do Python que representa um intervalo de tempo. Ele permite somar ou subtrair tempo de uma data.
        future_dates.append(date_pointer)
        
        # Mapeamento do Dia 
        month_v = date_pointer.month #Pega o número do mês (1 a 12)
        tavg = temp_annual_means.get(month_v, 22.0)  # Busca a temperatura média daquele mês no histórico, caso nao ache usa o padrao 22 graus
        hol  = int(cal.is_holiday(date_pointer.date())) #Verifica se a data é feriado
        
        # Lags do xgboost
        l_1 = hist_logs[-1]
        l_7 = hist_logs[-7]
        r_7 = np.mean(hist_logs[-7:])
        
        # xgboost full parameters
        isolated_day_matrix = pd.DataFrame([{
            'temp_mean_c': tavg,
            'is_holiday': hol,
            'day_of_week': date_pointer.dayofweek,
            'day_of_year': date_pointer.dayofyear,
            'month': month_v,
            'year': date_pointer.year,
            'lag_1': l_1,
            'lag_7': l_7,
            'rolling_mean_7': r_7
        }])
        
        # ML Prediction Loop-shot
        y_future_log = model.predict(isolated_day_matrix)[0] # modelo recebe os dados do dia que montados da matrix
        
        #Adciona o previsto no historico
        hist_logs.append(y_future_log)
        
        # converte a previsao que esta em log para escala natural
        y_future_real = np.exp(y_future_log)
        predictions.append(y_future_real)
        
    df_forecast = pd.DataFrame({'date': future_dates, 'predicted_mwh': predictions})
    return df_forecast

# Puxa o gatilho da Projeção Trimestral
df_forecast_3_months = run_recursive_future(final_model, df_ml, days_forward=90)





Initiating Auto-Regressive recursive chained prediction matrix (Next 90 days)...


In [0]:

def visualize_future(df_history, df_future):
    # Recortar o último meio ano de fatos reais p/ Gráfico limpo 
    df_recent = df_history.tail(180).copy()
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=df_recent['date'], y=df_recent['consumption_mwh_day'],
        mode='lines', name='Consolidated History',
        line=dict(color='#2196F3', width=2.5)  # Azul Brilhante
    ))
    
    fig.add_trace(go.Scatter(
        x=df_future['date'], y=df_future['predicted_mwh'],
        mode='lines', name='Estimated Future (XGBoost)',
        line=dict(color='#FF5722', width=2.5, dash='dash')  # Laranja Alerta Fatiado
    ))
    
    fig.update_layout(
        title='Operational MWh Projection - Databricks Machine Learning',
        xaxis_title='Timeline',
        yaxis_title='MWh / Day',
        template='plotly_dark',  # Padrão Tech Dinâmico
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    return fig

final_chart = visualize_future(df, df_forecast_3_months)
final_chart.show()
# display(final_chart)



In [0]:
print("Recording Predictions and History into the permanent Lakehouse layer...")

# Converter o Forecast de Pandas Preditivo para PySpark e gravar
df_spark_forecast = spark.createDataFrame(df_forecast_3_months)
df_spark_forecast.write.format("delta").mode("overwrite").saveAsTable("foodshop_predictions") #Delta:É um formato de armazenamento de dados baseado em arquivos (tipo Parquet), mas com recursos extras de banco de dados.

print("Pipeline Finished. Database successfully updated for reading by the Executive Dashboard Module (Script 3).")

Recording Predictions and History into the permanent Lakehouse layer...
Pipeline Finished. Database successfully updated for reading by the Executive Dashboard Module (Script 3).
